# 🏔️ Sistem Rekomendasi Rute Pendakian Gunung
## Content-Based Filtering dengan SBERT dan Cosine Similarity

**Paper:** "Sistem Rekomendasi Rute Pendakian Gunung Menggunakan Content-Based Filtering Berbasis SBERT dan Cosine Similarity"

---

### Komponen Utama:
1. **Data Collection & Fusion**: Ekstraksi fitur dari GPX + deskripsi manual
2. **Preprocessing**: Regex cleaning, case folding, stopword removal (selektif)
3. **Feature Extraction**: SBERT (paraphrase-multilingual-MiniLM-L12-v2) → 384 dimensi
4. **Similarity & Ranking**: Cosine Similarity dengan Top-N ranking

## Dokumentasi Fungsi Notebook

- `preprocess_text(text, remove_stopwords=True)`: membersihkan teks, melakukan case folding, dan menghapus stopword selektif tanpa stemming agar konteks tetap terbaca oleh SBERT.
- `parse_gpx(gpx_path)`: membaca file GPX dan mengambil titik lintasan berupa latitude, longitude, dan elevasi.
- `calculate_statistics(gpx, points)`: menghitung jarak, kenaikan elevasi, estimasi durasi Naismith, grade rata-rata, rentang elevasi, dan label kesulitan.
- `generate_narrative(stats, manual_description="")`: mengubah statistik numerik dan deskripsi manual menjadi narasi jalur yang bisa diproses embedding.
- `generate_embedding(text)`: mengubah teks menjadi embedding 384 dimensi. Saat SBERT tidak tersedia lokal, notebook memakai fallback deterministik agar demo tetap runnable.
- `calc_similarity(query_emb, route_emb)`: menghitung cosine similarity antara query pengguna dan embedding rute.
- `add_route(name, gpx_path, manual_description="")`: menjalankan pipeline ingest satu rute GPX sampai masuk ke database sementara notebook.
- `search_routes(query, top_n=5)`: memproses query natural language, membuat embedding query, menghitung skor kemiripan, dan mengembalikan rekomendasi teratas.

In [1]:
#@title 1️⃣ Install Dependencies
!pip install gpxpy sentence-transformers scikit-learn pandas numpy -q
print("✅ Dependencies installed!")

✅ Dependencies installed!



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#@title 2️⃣ Import Libraries
import os
import re
import json
import hashlib
import math
import numpy as np
import pandas as pd
import gpxpy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False
import warnings
PROJECT_ROOT = os.getcwd()
HF_CACHE_DIR = os.path.join(PROJECT_ROOT, 'storage', 'app', 'huggingface')
os.environ.setdefault('HF_HOME', HF_CACHE_DIR)
os.environ.setdefault('SENTENCE_TRANSFORMERS_HOME', os.path.join(HF_CACHE_DIR, 'sentence-transformers'))
os.environ.setdefault('TRANSFORMERS_CACHE', os.path.join(HF_CACHE_DIR, 'transformers'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


In [3]:
#@title 3️⃣ Text Preprocessing (Sesuai Paper)

STOPWORDS_ID = {
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'dengan', 'untuk', 'pada',
    'adalah', 'sebagai', 'dalam', 'juga', 'atau', 'ada', 'oleh', 'akan', 'sudah',
    'saya', 'kami', 'kita', 'mereka', 'dia', 'ia', 'anda', 'tersebut', 'dapat',
    'bisa', 'harus', 'telah', 'lalu', 'kemudian', 'serta', 'maupun', 'saat',
    'ketika', 'bila', 'kalau', 'jika', 'karena', 'agar', 'supaya', 'hingga',
    'sampai', 'antara', 'seperti', 'yaitu', 'yakni', 'bahwa', 'namun', 'tetapi'
}

PRESERVE_WORDS = {
    'tidak', 'bukan', 'jangan', 'belum', 'tanpa',
    'mudah', 'sulit', 'curam', 'landai', 'panjang', 'pendek',
    'tinggi', 'rendah', 'sejuk', 'panas', 'dingin', 'indah', 'bagus',
    'pemula', 'berpengalaman', 'santai', 'menantang', 'ekstrem'
}

def preprocess_text(text, remove_stopwords=True):
    if not text:
        return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s\-]', ' ', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    if remove_stopwords:
        words = text.split()
        filtered_words = [w for w in words if w in PRESERVE_WORDS or w not in STOPWORDS_ID]
        text = ' '.join(filtered_words)
    return text

print("✅ Preprocessing functions defined!")

✅ Preprocessing functions defined!


In [4]:
#@title 4️⃣ GPX Processing Functions

def parse_gpx(gpx_path):
    with open(gpx_path, 'r', encoding='utf-8') as gpx_file:
        gpx = gpxpy.parse(gpx_file)
    points = []
    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                points.append({
                    'lat': point.latitude,
                    'lon': point.longitude,
                    'ele': point.elevation if point.elevation else 0
                })
    return gpx, points

def calculate_statistics(gpx, points):
    distance_km = gpx.length_3d() / 1000.0
    elevation_gain = sum(max(0, points[i]['ele'] - points[i-1]['ele']) for i in range(1, len(points)))
    naismith_duration = (distance_km / 5) + (elevation_gain / 600)
    average_grade = (elevation_gain / (distance_km * 1000)) * 100 if distance_km > 0 else 0
    
    if average_grade < 5: difficulty = "mudah"
    elif average_grade < 10: difficulty = "sedang"
    elif average_grade < 15: difficulty = "sulit"
    else: difficulty = "sangat sulit"
    
    elevations = [p['ele'] for p in points if p['ele'] > 0]
    return {
        'distance_km': round(distance_km, 2),
        'elevation_gain_m': int(elevation_gain),
        'naismith_duration_hour': round(naismith_duration, 2),
        'average_grade_pct': round(average_grade, 2),
        'min_elevation': int(min(elevations)) if elevations else 0,
        'max_elevation': int(max(elevations)) if elevations else 0,
        'difficulty': difficulty
    }

def generate_narrative(stats, manual_description=""):
    jarak_desc = "pendek" if stats['distance_km'] < 3 else "sedang" if stats['distance_km'] < 7 else "panjang"
    waktu_desc = "singkat" if stats['naismith_duration_hour'] < 2 else "sedang" if stats['naismith_duration_hour'] < 4 else "lama"
    
    if stats['average_grade_pct'] < 5: grade_desc = "landai dan ramah pemula"
    elif stats['average_grade_pct'] < 10: grade_desc = "menantang dengan kemiringan sedang"
    elif stats['average_grade_pct'] < 15: grade_desc = "curam dan membutuhkan stamina yang baik"
    else: grade_desc = "sangat curam dan berbahaya"
    
    narrative = f"Jalur pendakian dengan jarak {jarak_desc} sekitar {stats['distance_km']} kilometer. Estimasi waktu tempuh {waktu_desc} selama {stats['naismith_duration_hour']} jam. Karakteristik jalur {grade_desc} dengan grade rata-rata {stats['average_grade_pct']}%. Total kenaikan elevasi {stats['elevation_gain_m']} meter. Tingkat kesulitan: {stats['difficulty']}."
    
    if manual_description:
        narrative = manual_description + " " + narrative
    return narrative

print("✅ GPX processing functions defined!")

✅ GPX processing functions defined!


In [5]:
#@title 5️⃣ Load SBERT Model
print("⏳ Loading SBERT model...")
model_cache = os.path.join(HF_CACHE_DIR, 'sentence-transformers', 'models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2')
refs_main = os.path.join(model_cache, 'refs', 'main')
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
if os.path.exists(refs_main):
    with open(refs_main, 'r', encoding='utf-8') as f:
        revision = f.read().strip()
    snapshot = os.path.join(model_cache, 'snapshots', revision)
    if os.path.exists(snapshot):
        model_name = snapshot
try:
    model = SentenceTransformer(model_name, local_files_only=True)
    print(f"✅ Model loaded! Embedding dimension: {model.get_sentence_embedding_dimension()}")
except Exception as exc:
    print(f"⚠️ SBERT unavailable locally, using deterministic demo embedding: {exc}")
    class HashEmbeddingModel:
        def get_sentence_embedding_dimension(self):
            return 384
        def encode(self, text):
            vector = np.zeros(384, dtype=float)
            for token in str(text).split():
                digest = hashlib.sha256(token.encode('utf-8')).digest()
                vector[int.from_bytes(digest[:2], 'little') % 384] += 1.0
            norm = np.linalg.norm(vector)
            return vector / norm if norm else vector
    model = HashEmbeddingModel()
    print(f"✅ Fallback embedding ready! Dimension: {model.get_sentence_embedding_dimension()}")

⏳ Loading SBERT model...
⚠️ SBERT unavailable locally, using deterministic demo embedding: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory C:\laragon\www\rutestripV2\colab\storage\app\huggingface\sentence-transformers\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2\snapshots\e8f8c211226b894fcb81acc59f3b34ba3efd5f42.
✅ Fallback embedding ready! Dimension: 384


In [6]:
#@title 6️⃣ Core Functions

routes_database = []

def generate_embedding(text):
    return model.encode(text).tolist()

def calc_similarity(query_emb, route_emb):
    return float(cosine_similarity(
        np.array(query_emb).reshape(1, -1),
        np.array(route_emb).reshape(1, -1)
    )[0][0])

def add_route(name, gpx_path, manual_description=""):
    try:
        gpx, points = parse_gpx(gpx_path)
        if not points: return None
        stats = calculate_statistics(gpx, points)
        narrative = generate_narrative(stats, manual_description)
        embedding = generate_embedding(preprocess_text(narrative, False))
        route = {
            'id': len(routes_database) + 1,
            'name': name,
            'stats': stats,
            'narrative_text': narrative,
            'embedding': embedding
        }
        routes_database.append(route)
        print(f"✅ Added: {name} | {stats['distance_km']}km | {stats['difficulty']}")
        return route
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def search_routes(query, top_n=5):
    if not routes_database: return []
    processed = preprocess_text(query, True)
    query_emb = generate_embedding(processed)
    results = [{'route': r, 'similarity': calc_similarity(query_emb, r['embedding'])} for r in routes_database]
    results.sort(key=lambda x: x['similarity'], reverse=True)
    return results[:top_n]

print("✅ Core functions defined!")

✅ Core functions defined!


In [7]:
#@title 7️⃣ Upload GPX Files
print("📁 Upload file GPX Anda:")
uploaded = files.upload() if IN_COLAB else {}
if not IN_COLAB:
    print("ℹ️ Local run detected, skipping upload and using sample data in the next cell.")

for filename in uploaded.keys():
    if filename.endswith(('.gpx', '.xml')):
        name = filename.replace('.gpx', '').replace('.xml', '').replace('_', ' ').title()
        desc = input(f"Deskripsi untuk {name} (Enter untuk skip): ")
        add_route(name, filename, desc)

print(f"\n📚 Total: {len(routes_database)} rute")

📁 Upload file GPX Anda:
ℹ️ Local run detected, skipping upload and using sample data in the next cell.

📚 Total: 0 rute


In [8]:
#@title 8️⃣ Demo dengan Data Sample (jika tidak upload GPX)
if not routes_database:
    samples = [
        ('Gunung Prau', 'Pemandangan sunrise, padang rumput', 3.5, 450, 'mudah'),
        ('Gunung Merbabu', 'Sabana luas, sumber air di pos 2', 12.0, 1200, 'sulit'),
        ('Gunung Sindoro', 'Jalur curam, hutan pinus', 8.0, 1400, 'sangat sulit'),
        ('Bukit Sikunir', 'Jalur pendek sunrise', 1.5, 150, 'mudah'),
        ('Gunung Sumbing', 'Trek panjang, kawah, air terjun', 10.0, 1800, 'sangat sulit'),
    ]
    for name, desc, dist, ele, diff in samples:
        stats = {
            'distance_km': dist, 'elevation_gain_m': ele,
            'naismith_duration_hour': round((dist/5) + (ele/600), 2),
            'average_grade_pct': round((ele/(dist*1000))*100, 2),
            'min_elevation': 1500, 'max_elevation': 2500 + ele, 'difficulty': diff
        }
        narrative = generate_narrative(stats, desc)
        routes_database.append({
            'id': len(routes_database) + 1, 'name': name, 'stats': stats,
            'narrative_text': narrative,
            'embedding': generate_embedding(preprocess_text(narrative, False))
        })
        print(f"✅ {name}")
    print(f"\n📚 {len(routes_database)} sample routes loaded")

✅ Gunung Prau
✅ Gunung Merbabu
✅ Gunung Sindoro
✅ Bukit Sikunir
✅ Gunung Sumbing

📚 5 sample routes loaded


In [9]:
#@title 9️⃣ Pencarian dengan Natural Language
query = input("🔍 Masukkan preferensi (contoh: 'jalur mudah untuk pemula'): ") if IN_COLAB else "jalur mudah untuk pemula"

if query.strip():
    results = search_routes(query, 5)
    print("\n" + "="*60)
    print("📊 HASIL REKOMENDASI")
    print("Formula: Sim(Q,D) = (Q·D) / (||Q||×||D||)")
    print("="*60)
    for i, r in enumerate(results, 1):
        route = r['route']
        print(f"\n#{i} {route['name']}")
        print(f"   📏 {route['stats']['distance_km']}km | ⛰️ {route['stats']['elevation_gain_m']}m | 🏷️ {route['stats']['difficulty']}")
        print(f"   📊 Similarity: {r['similarity']:.4f} ({r['similarity']*100:.1f}%)")
        print(f"   🤖 {route['narrative_text'][:100]}...")


📊 HASIL REKOMENDASI
Formula: Sim(Q,D) = (Q·D) / (||Q||×||D||)

#1 Bukit Sikunir
   📏 1.5km | ⛰️ 150m | 🏷️ mudah
   📊 Similarity: 0.3405 (34.1%)
   🤖 Jalur pendek sunrise Jalur pendakian dengan jarak pendek sekitar 1.5 kilometer. Estimasi waktu tempu...

#2 Gunung Prau
   📏 3.5km | ⛰️ 450m | 🏷️ mudah
   📊 Similarity: 0.2705 (27.1%)
   🤖 Pemandangan sunrise, padang rumput Jalur pendakian dengan jarak sedang sekitar 3.5 kilometer. Estima...

#3 Gunung Sindoro
   📏 8.0km | ⛰️ 1400m | 🏷️ sangat sulit
   📊 Similarity: 0.2500 (25.0%)
   🤖 Jalur curam, hutan pinus Jalur pendakian dengan jarak panjang sekitar 8.0 kilometer. Estimasi waktu ...

#4 Gunung Merbabu
   📏 12.0km | ⛰️ 1200m | 🏷️ sulit
   📊 Similarity: 0.1761 (17.6%)
   🤖 Sabana luas, sumber air di pos 2 Jalur pendakian dengan jarak panjang sekitar 12.0 kilometer. Estima...

#5 Gunung Sumbing
   📏 10.0km | ⛰️ 1800m | 🏷️ sangat sulit
   📊 Similarity: 0.1684 (16.8%)
   🤖 Trek panjang, kawah, air terjun Jalur pendakian dengan jarak panja

In [10]:
#@title 🔟 Test Berbagai Query
test_queries = [
    "jalur mudah untuk pemula",
    "trek menantang elevasi tinggi",
    "pendakian singkat 2-3 jam",
    "jalur landai sabana sunrise"
]

for q in test_queries:
    print(f"\n🔍 Query: '{q}'")
    for i, r in enumerate(search_routes(q, 3), 1):
        print(f"   {i}. {r['route']['name']} - {r['similarity']:.4f}")


🔍 Query: 'jalur mudah untuk pemula'
   1. Bukit Sikunir - 0.3405
   2. Gunung Prau - 0.2705
   3. Gunung Sindoro - 0.2500

🔍 Query: 'trek menantang elevasi tinggi'
   1. Gunung Sumbing - 0.1459
   2. Gunung Prau - 0.0781
   3. Gunung Merbabu - 0.0762

🔍 Query: 'pendakian singkat 2-3 jam'
   1. Gunung Prau - 0.2343
   2. Bukit Sikunir - 0.2212
   3. Gunung Merbabu - 0.1525

🔍 Query: 'jalur landai sabana sunrise'
   1. Bukit Sikunir - 0.2949
   2. Gunung Prau - 0.2343
   3. Gunung Merbabu - 0.2287


In [11]:
#@title 1️⃣1️⃣ Export ke CSV
df = pd.DataFrame([{
    'ID': r['id'], 'Nama': r['name'],
    'Jarak_km': r['stats']['distance_km'],
    'Elevasi_m': r['stats']['elevation_gain_m'],
    'Durasi_jam': r['stats']['naismith_duration_hour'],
    'Grade_%': r['stats']['average_grade_pct'],
    'Kesulitan': r['stats']['difficulty'],
    'Narrative': r['narrative_text']
} for r in routes_database])

df.to_csv('hiking_routes.csv', index=False)
if IN_COLAB:
    files.download('hiking_routes.csv')
else:
    print("ℹ️ Local run: hiking_routes.csv saved in the current working directory.")
print("✅ Exported!")

ℹ️ Local run: hiking_routes.csv saved in the current working directory.
✅ Exported!
